# Analysis

In [ ]:
# import libraries
import ast
import numpy as np
import pandas as pd
import ast
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import json
import re
import os
from google.colab import drive
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import gaussian_kde
import numpy as np
import pandas as pd
from scipy.spatial import ConvexHull
import ast

In [ ]:
# Mount Google Drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# load preprocessed data
df = pd.read_excel('/content/drive/MyDrive/Thesis/analysis/df_final_analysis/df_final_analysis2.xlsx')

# filter orginal and generated df
df_original_image = df[df['original_image']==True]
df_generated_image = df[df['original_image']==False]

## Plot number of original and generated images per politician

In [ ]:
plt.rcParams.update({
    "font.family": "Times New Roman",
    "font.size": 12,
    "axes.edgecolor": "black",
    "axes.linewidth": 1,
    "axes.titlesize": 12,
    "axes.labelsize": 12,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "legend.fontsize": 11
})

summary_df = pd.DataFrame(data)
pivot_df = summary_df.pivot(index="politician", columns="image_type", values="total_images").reset_index()
fig, ax = plt.subplots(figsize=(6, 4))

bar_width = 0.35
index = range(len(pivot_df))

ax.bar([i - bar_width / 2 for i in index], pivot_df["original"], width=bar_width, label="Original", color="gray")
ax.bar([i + bar_width / 2 for i in index], pivot_df["generated"], width=bar_width, label="Generated", color="black")

ax.set_xlabel("Politician")
ax.set_ylabel("# of images")
ax.set_title("Included images")
ax.set_xticks(index)
ax.set_xticklabels(pivot_df["politician"].str.title())
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.legend(frameon=False)
plt.tight_layout()
plt.show()


## Plot dispersion distances of centroids

In [ ]:
def calculate_dispersion_metrics(df, columns, df_name="Unnamed DF"):
    results = []
    for col in columns:
        raw_points = df[col].dropna()

        def to_array(p):
            try:
                if isinstance(p, str):
                    p = ast.literal_eval(p)
                return np.array(p)
            except Exception as e:
                print(f"Error parsing column {col}, point {p}: {e}")
                return None

        # Apply function
        points = raw_points.apply(to_array).dropna().tolist()
        points = np.array(points)

        # check for too little points
        if len(points) < 2:
            print(f"Skipping column {col} in DataFrame {df_name}, not enough valid points.")
            continue

        centroid = np.mean(points, axis=0)
        distances = np.linalg.norm(points - centroid, axis=1)
        avg_distance = np.mean(distances)

        # store the results
        results.append({
            "df_name": df_name,
            "column_name": col,
            "num_points": len(points),
            "avg_distance_to_centroid": avg_distance,
            "centroid_x": centroid[0],
            "centroid_y": centroid[1]
        })

    results_df = pd.DataFrame(results)
    return results_df



# Define df names
df_names = ['df_original_image', 'df_generated_image']
df = df.rename(columns={'image_path_x': 'image_path'})
df_original_image= df_original_image.rename(columns={'image_path_x': 'image_path'})
df_generated_image= df_generated_image.rename(columns={'image_path_x': 'image_path'})
dfs = [df_original_image, df_generated_image]

# create new df to store results
all_results = pd.DataFrame()

body_part_columns = ['center_face_donald_trump',
       'LEFT_SHOULDER_donald_trump', 'RIGHT_SHOULDER_donald_trump',
       'LEFT_WRIST_donald_trump', 'RIGHT_WRIST_donald_trump']

# run calculator
for idx, df in enumerate(dfs):
    df_name = df_names[idx]
    result_df = calculate_dispersion_metrics(df, body_part_columns, df_name)
    all_results = pd.concat([all_results, result_df], ignore_index=True)

print(all_results)

## Plot Surfaces distributions

In [ ]:
df_surfaces = df[['surface_of_donald_trump_box', 'surface_of_angela_merkel_box',
       'surface_of_mark_rutte_box', 'surface_of_barack_obama_box']]

In [ ]:
bin_edges = np.linspace(0, 200000, 21)
x_kde = np.linspace(0, 200000, 1000)

# Plot 2x2 histogram
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

for i, column in enumerate(df_surfaces.columns):
    data = df_surfaces[column].dropna()

    # histogram
    axes[i].hist(data, bins=bin_edges, color='gray', edgecolor='black', linewidth=0.3, density=True)

    # KDE lijn
    kde = gaussian_kde(data)
    axes[i].plot(x_kde, kde(x_kde), color='black', linewidth=1.2)

    axes[i].set_title(column.replace('surface_of_', '').replace('_box', '').replace('_', ' ').title())
    axes[i].set_xlabel('Surface Area')
    axes[i].set_ylabel('Density')
    axes[i].set_xlim(0, 200000)

plt.tight_layout()
plt.show()


## Plot Face Centers

### Face centers original

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import ast
import os

def plot_multiple_feature_points(df, feature_cols, title, point_color='red'):
    df = df.head(20)
    num_images = len(df)
    num_cols = 5
    num_rows = int(np.ceil(num_images / num_cols))

    fig, axs = plt.subplots(num_rows, num_cols, figsize=(4*num_cols, 4*num_rows))
    axs = axs.flatten()

    for idx, (_, row) in enumerate(df.iterrows()):
        img_path = row['image_path']
        if not os.path.exists(img_path):
            axs[idx].axis('off')
            axs[idx].set_title("Image not found")
            continue

        # Load and display the image
        img = mpimg.imread(img_path)
        axs[idx].imshow(img)
        axs[idx].axis('off')

        # Plot each feature point
        for col in feature_cols:
            point = row.get(col)
            if isinstance(point, str):
                try:
                    point = ast.literal_eval(point)
                except:
                    point = None

            if isinstance(point, list) and len(point) == 2:
                x = point[0] * img.shape[1]
                y = point[1] * img.shape[0]
                axs[idx].scatter(x, y, color=point_color, s=50)

        axs[idx].set_title(os.path.basename(img_path), fontsize=10)

    # Turn off any unused subplots
    for i in range(idx + 1, len(axs)):
        axs[i].axis('off')

    fig.suptitle(title, fontsize=16)
    plt.tight_layout()
    plt.show()


In [ ]:
politicians_list = ['barack_obama', 'donald_trump', 'angela_merkel', 'mark_rutte']

plt.style.use('grayscale')

# 2x2 plots
fig, axes = plt.subplots(2, 2, figsize=(10, 10))
fig.patch.set_facecolor('white')

axes = axes.flatten()

for ax, politician in zip(axes, politicians_list):
    ax.set_facecolor('white')

    plot_multiple_feature_points(
        df_original_image,
        [f'center_face_{politician}'],
        f"Generated {politician} images: Face",
        ax=ax
    )

    ax.set_title(f"{politician.replace('_', ' ').title()}", fontsize=10)
    ax.set_xticks([])
    ax.set_yticks([])
plt.tight_layout()
plt.show()


### Face Centers Generated

In [ ]:
import matplotlib.pyplot as plt

politicians_list = ['barack_obama', 'donald_trump', 'angela_merkel', 'mark_rutte']

plt.style.use('grayscale')

# 2x2 plots
fig, axes = plt.subplots(2, 2, figsize=(10, 10))

fig.patch.set_facecolor('white')

axes = axes.flatten()

for ax, politician in zip(axes, politicians_list):
    ax.set_facecolor('white')

    plot_multiple_feature_points(
        df_generated_image,
        [f'center_face_{politician}'],
        f"Generated {politician} images: Face",
        ax=ax
    )

    ax.set_title(f"{politician.replace('_', ' ').title()}", fontsize=10)
    ax.set_xticks([])
    ax.set_yticks([])

plt.tight_layout()
plt.show()


## Plot body parts

### Plot body parts orginal

In [ ]:
import matplotlib.pyplot as plt
from ast import literal_eval
import matplotlib.cm as cm
import numpy as np

def plot_multiple_feature_points(df, columns, title, ax=None):
    def parse_coords(x):
        if isinstance(x, str):
            try:
                return literal_eval(x)
            except:
                return None
        elif isinstance(x, (list, tuple)) and len(x) == 2:
            return x
        return None

    img_size = 1024  # Set standard image size

    if ax is None:
        fig, ax = plt.subplots(figsize=(8, 8))

    # Generate colors for each column
    colors = cm.get_cmap('tab10', len(columns))

    for idx, column in enumerate(columns):
        temp_df = df.copy()
        temp_df['coords'] = temp_df[column].apply(parse_coords)
        temp_df = temp_df.dropna(subset=['coords'])
        temp_df['x'] = temp_df['coords'].apply(lambda c: c[0] * img_size)
        temp_df['y'] = temp_df['coords'].apply(lambda c: c[1] * img_size)
        ax.scatter(temp_df['x'], temp_df['y'], color=colors(idx), s=10, label=column)

    ax.invert_yaxis()
    ax.set_xlim(0, img_size)
    ax.set_ylim(img_size, 0)
    ax.set_title(title, fontsize=10)
    ax.set_xlabel("X", fontsize=8)
    ax.set_ylabel("Y", fontsize=8)
    ax.grid(True, color='gray', linestyle='--', linewidth=0.5)
    ax.legend(fontsize=6, loc='upper right')


politicians_list = ['barack_obama', 'donald_trump', 'angela_merkel', 'mark_rutte']

# Define landmarks
features = [
    'NOSE',
    'LEFT_WRIST', 'RIGHT_WRIST',
    'LEFT_ELBOW', 'RIGHT_ELBOW',
    'LEFT_SHOULDER', 'RIGHT_SHOULDER'
]

plt.style.use('grayscale')

# 2x2 plots
fig, axes = plt.subplots(2, 2, figsize=(12, 12))

fig.patch.set_facecolor('white')

axes = axes.flatten()

for ax, politician in zip(axes, politicians_list):
    ax.set_facecolor('white')

    feature_columns = [f'{feature}_{politician}' for feature in features]

    plot_multiple_feature_points(
        df_original_image,
        feature_columns,
        f"Generated {politician} images: Keypoints",
        ax=ax
    )

    ax.set_title(f"{politician.replace('_', ' ').title()}", fontsize=10)
    ax.set_xticks([])
    ax.set_yticks([])

plt.tight_layout()
plt.show()


### Plot body parts generated

In [ ]:
import matplotlib.pyplot as plt
from ast import literal_eval
import matplotlib.cm as cm
import numpy as np

def plot_multiple_feature_points(df, columns, title, ax=None):
    def parse_coords(x):
        if isinstance(x, str):
            try:
                return literal_eval(x)
            except:
                return None
        elif isinstance(x, (list, tuple)) and len(x) == 2:
            return x
        return None

    img_size = 1024  # image size

    if ax is None:
        fig, ax = plt.subplots(figsize=(8, 8))

    # colors for each column
    colors = cm.get_cmap('tab10', len(columns))

    for idx, column in enumerate(columns):
        temp_df = df.copy()
        temp_df['coords'] = temp_df[column].apply(parse_coords)
        temp_df = temp_df.dropna(subset=['coords'])
        temp_df['x'] = temp_df['coords'].apply(lambda c: c[0] * img_size)
        temp_df['y'] = temp_df['coords'].apply(lambda c: c[1] * img_size)
        ax.scatter(temp_df['x'], temp_df['y'], color=colors(idx), s=10, label=column)

    ax.invert_yaxis()
    ax.set_xlim(0, img_size)
    ax.set_ylim(img_size, 0)
    ax.set_title(title, fontsize=10)
    ax.set_xlabel("X", fontsize=8)
    ax.set_ylabel("Y", fontsize=8)
    ax.grid(True, color='gray', linestyle='--', linewidth=0.5)
    ax.legend(fontsize=6, loc='upper right')

politicians_list = ['barack_obama', 'donald_trump', 'angela_merkel', 'mark_rutte']

features = [
    'NOSE',
    'LEFT_WRIST', 'RIGHT_WRIST',
    'LEFT_ELBOW', 'RIGHT_ELBOW',
    'LEFT_SHOULDER', 'RIGHT_SHOULDER'
]

plt.style.use('grayscale')

# 2x2 plots
fig, axes = plt.subplots(2, 2, figsize=(12, 12))

fig.patch.set_facecolor('white')

axes = axes.flatten()

for ax, politician in zip(axes, politicians_list):
    ax.set_facecolor('white')

    feature_columns = [f'{feature}_{politician}' for feature in features]

    plot_multiple_feature_points(
        df_generated_image,
        feature_columns,
        f"Generated {politician} images: Keypoints",
        ax=ax
    )

    ax.set_title(f"{politician.replace('_', ' ').title()}", fontsize=10)
    ax.set_xticks([])
    ax.set_yticks([])

plt.tight_layout()
plt.show()


## Plot iconicity boxplots

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.set(style="whitegrid")
plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman"],
    "axes.titlesize": 14,
    "axes.labelsize": 12,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11
})

plt.figure(figsize=(8, 6))
ax = sns.boxplot(data=df, x='subject_politician', y='Iconicity_z',
                 color='white', fliersize=3, linewidth=1.2)

sns.despine(trim=True)
ax.set_title("Iconicity scores by politician", pad=15)
ax.set_xlabel("Politician")
ax.set_ylabel("Iconicity score")

plt.tight_layout()
plt.show()

